<h1 align = "Center">
Lotus Group Customer Behavior Analysis

<h2 align = "Center">
Feature Engineering and Target Construction
</h1>

This is one of the most important phases of the project.

Our two modelling objectives are:

**Classification**: Predict whether a customer will make at least one completed purchase in the next 90 days.

**Regression**: Predict how much a customer will spend on completed purchases in the next 90 days.
An important modelling decision

We don't want to simply use the entire historical customer spending table as our model input.

For example, suppose we want to predict customer behaviour as at 31 December 2023.

Our model must use information available on or before that date. Purchases made in 2024 must be used to establish the prediction targets, not the historical features.

To achieve this, we want to use quarterly observation dates.

For instance:

|Observation date	|Historical features	|Future target period|
|---|---|---|
|31 March 2022	|Data up to 31 March 2022	|Following 90 days|
|30 June 2022	|Data up to 30 June 2022	|Following 90 days|
|30 September 2022	|Data up to 30 September 2022	|Following 90 days|
|...	|...	|...|
|30 June 2024	|Data up to 30 June 2024	|Following 90 days|
|30 September 2024	|Data up to 30 September 2024	|Following 90 days|


* This should give us multiple historical observations per customer rather than relying on just one snapshot. 

* It should also allow us to evaluate the models chronologically.

We want to retain all 3,000 registered customers in each observation period, including customers who have never purchased. 

- Should be important because identifying potential first-time buyers is also commercially valuable.

We'll start by constructing the observation-date framework.

# Establishing Observation Dates

Our dataset covers 1st January 2022 to 31 December 2024.

We want to use quarterly observation dates, beginning on 31st March 2022.

**Why start in March?** Because it gives us an initial three months of transaction history before the first observation date.

Also, we want to stop at 30th September 2024 because we need a complete 90-day period after every observation date. Our dataset does not extend beyond 31st December 2024.

In [ ]:
import pandas as pd
# Generate quarterly observation dates
snapshot_dates = pd.date_range(
    start="2022-03-31",
    end="2024-09-30",
    freq="QE"
)

# Convert into a DataFrame

snapshots = pd.DataFrame({
    "snapshot_date": snapshot_dates
})

# Calculate the end of each 90-day prediction window

snapshots["target_end_date"] = (
    snapshots["snapshot_date"] + pd.Timedelta(days=90)
)

snapshots

#### Our observation framework

The final observation date is 30th September 2024, with a target window ending 29th December 2024.

This is valid because the dataset extends to 31st December 2024, giving us the complete 90-day outcome period.

We wnt to use the following convention consistently:
- Historical transactions: `order_date <= snapshot_date`
- Future transactions: `order_date > snapshot_date` and `order_date <= target_end_date`

This prevents information from the future from entering our historical customer features.

### Creating Customer Snapshot Framework

Now we create a dataset containing every registered customer at every observation date.

We have:
- 3,000 registered customers.
- 11 observation dates.
- 33,000 expected customer-observation records.

**Why include all customers?**

We want the models to learn about:
- Existing customers who purchase frequently.
- Customers whose purchasing activity has declined.
- Customers who have never purchased.
- Customers who have registered but have not yet made a completed purchase.

In [ ]:
from pathlib import Path

# Define path to data folder
data_path = Path(r"../01_data/processed")

def load_csv (filename):
    return pd.read_csv(data_path/filename)

In [ ]:
# Extract customer dataset
customers_clean = load_csv("customers_clean.csv")

# Create every possible customer-observation combination
customer_snapshots = (
    customers_clean.assign(key=1)
    .merge(
        snapshots.assign(key=1),
        on="key"
    )
    .drop(columns="key")
)

# Retain only customers registered by each observation date

customer_snapshots = customer_snapshots[
    customer_snapshots["registration_date"]
    <= customer_snapshots["snapshot_date"]
].copy()

# Sort the records

customer_snapshots = customer_snapshots.sort_values(
    ["snapshot_date", "customer_id"]
).reset_index(drop=True)

# Validate

print("Total records:", len(customer_snapshots))

print(
    "Unique customers:",
    customer_snapshots["customer_id"].nunique()
)

print(
    "Unique observation dates:",
    customer_snapshots["snapshot_date"].nunique()
)

print(
    "Invalid registration dates:",
    (
        customer_snapshots["registration_date"]
        > customer_snapshots["snapshot_date"]
    ).sum()
)

# Inspect the resulting dataset

print("Total records:", len(customer_snapshots))
print("Unique customers:", customer_snapshots["customer_id"].nunique())
print("Unique observation dates:", customer_snapshots["snapshot_date"].nunique())

customer_snapshots.head(10)

We now have 27,548 customer-observation records representing 3,000 customers across 11 quarterly observation dates.

Our customer snapshot framework is ready.

## Calculating Historical Purchasing Features

We will begin with four fundamental customer behaviour features:
1. Recency: Days since the customer's last completed purchase.
2. Frequency: Number of completed orders placed before or on the observation date.
3. Monetary: Total amount spent on completed orders before or on the observation date.
4. Average order value (AOV): Average value of the customer's completed orders.

These are commonly known as RFM-related features. They are useful because they summarise how recently, frequently, and how much a customer purchases.

**Important consideration**

We want to calculate these features separately for every observation date.

For example, when calculating a customer's features as at 30th June 2023, we must exclude purchases made after 30th June 2023, even though those transactions exist in our dataset.

### Calculating historical features

We'll start by creating a function that calculates these four features for a given observation date.

In [ ]:
# Define completed orders data frame
orders = load_csv("orders_clean.csv")

completed_orders = orders[
    orders["order_status"] == "Completed"
].copy()

def calculate_customer_features(snapshot_date):

    # Retain only completed orders available by the observation date
    historical_orders = completed_orders[
        completed_orders["order_date"] <= snapshot_date
    ].copy()

    # Calculate customer-level historical purchasing behaviour
    customer_features = (
        historical_orders
        .groupby("customer_id")
        .agg(
            last_purchase_date=("order_date", "max"),
            total_orders=("order_id", "nunique"),
            total_spending=("total_revenue", "sum"),
            average_order_value=("total_revenue", "mean")
        )
        .reset_index()
    )

    # Calculate recency
    customer_features["recency_days"] = (
        snapshot_date - customer_features["last_purchase_date"]
    ).dt.days

    # Remove the intermediate date column
    customer_features = customer_features.drop(
        columns="last_purchase_date"
    )

    return customer_features

### Testing the Function

Before we apply it to all 11 observation dates, let's test it using the first observation date.

In [ ]:
completed_orders["order_date"] = pd.to_datetime(
    completed_orders["order_date"],
    errors = "coerce"
)

customer_snapshots["snapshot_date"] = pd.to_datetime(
    customer_snapshots["snapshot_date"],
    errors = "coerce"
)

test_features = calculate_customer_features(
    snapshots.loc[0, "snapshot_date"]
)

print("Observation date:", snapshots.loc[0, "snapshot_date"])
print("Customers with completed purchases:",
      test_features["customer_id"].nunique())

test_features.head(10)

733 customers had made at least one completed purchase by 31st March 2022.

We are not discarding the remaining 2,267 customers. 
- They may be new customers, customers who have not purchased yet, or customers whose recorded orders were cancelled or returned.

We want to preserve all of them in the final modelling dataset.

### Applying Historical Features to All Observation Dates

With the function working for the first observation date, let's apply it across all 11 dates.

In [ ]:
# Calculate historical purchasing features for every observation date

historical_features = pd.concat(
    [
        calculate_customer_features(snapshot_date)
        .assign(snapshot_date=snapshot_date)
        for snapshot_date in snapshots["snapshot_date"]
    ],
    ignore_index=True
)

print("Total historical feature records:", len(historical_features))
print("Unique customers:", historical_features["customer_id"].nunique())
print("Unique observation dates:", historical_features["snapshot_date"].nunique())

historical_features.head(10)

#### Findings

The output is reasonable.

We have:

|Metric	|Result|
|---|---:|
|Total customer-observation records	|33,000|
|Records with historical completed purchases	|24,353|
|Unique customers with at least one historical completed purchase	|2,868|
|Observation dates	|11|


The difference between 33,000 and 24,353 represents customer-observation periods in which customers had not yet made a completed purchase.

Notice that 2,868 is the number of distinct customers who had purchased at least once by at least one observation date. It does not mean only 2,868 customers were active at every observation date.

### Merging Historical Features Into the Full Customer Snapshots

We want to bring the historical features into customer_snapshots.

We want to use a left join so that no customer-observation record is lost.

In [ ]:
# Merge historical purchasing features into the full customer snapshots

modeling_data = customer_snapshots.merge(
    historical_features,
    on=["customer_id", "snapshot_date"],
    how="left",
    validate="one_to_one"
)

print("Total records:", len(modeling_data))
print("Unique customers:", modeling_data["customer_id"].nunique())
print("Unique observation dates:", modeling_data["snapshot_date"].nunique())

modeling_data.head(10)

#### Handling Customers Without Historical Purchases

For customers who have not made a completed purchase by an observation date:
- total_orders should be 0.
- total_spending should be 0.
- average_order_value should be 0.
- recency_days should remain missing for now.

> A customer who has never purchased does not have a meaningful last purchase date. Assigning zero recency would incorrectly suggest that they purchased today.

In [ ]:
# Replace missing historical purchasing counts and amounts with zero

feature_columns = [
    "total_orders",
    "total_spending",
    "average_order_value"
]

modeling_data[feature_columns] = (
    modeling_data[feature_columns].fillna(0)
)

modeling_data.head(10)

#### Result Validation

In [ ]:
print("Total modelling records:", len(modeling_data))

print("Missing values in historical features:")
print(modeling_data[
    ["recency_days", "total_orders", "total_spending", "average_order_value"]
].isna().sum())

print("\nCustomers with no historical completed purchases:")
print((modeling_data["total_orders"] == 0).sum())

#### **What the output tells us**:
* 27,548 modelling records: These are eligible customer–snapshot observations after excluding customers who had not yet registered.
* 6,098 observations without previous completed purchases: These are registered customers who had not made a completed purchase by the relevant snapshot date.
* 6,098 missing `recency_days` values: This is expected. A customer with no previous completed purchase has no last-purchase date from which to calculate recency.
* Zero missing values in the other historical features: Filling total_orders, total_spending, and `average_order_value` with zero was appropriate.

We want to retain the missing `recency_days` values for now. Later, when preparing the modelling data, we can handle them appropriately.

### Adding Customer Tenure

Customer tenure measures how long a customer has been registered with Lotus Group as of the observation date.

- It is useful because we wouldn't want to judge a customer registered two weeks ago by the same purchasing expectations as someone registered two years ago.

In [ ]:
# Parse snapshot_date and registration_date
date_cols = ["snapshot_date","registration_date"]
modeling_data[date_cols] = modeling_data[date_cols].apply(pd.to_datetime,
    errors = "coerce"
)

modeling_data["customer_tenure_days"] = (
    modeling_data["snapshot_date"]
    - modeling_data["registration_date"]
).dt.days